# Modeling Part 1 - Brian 2 tutorial

This notebook contains a tutorial on creating a running model with the Brian 2 package, based on the official [Brian 2 tutorials](https://brian2.readthedocs.io/en/stable/resources/tutorials/index.html).

[Brian 2](https://briansimulator.org) is a Python simulator for networks of spiking neurons (Stimberg et al., 2019). Its main idea is that you describe a model the way you would write it in a paper, as **differential equations with physical units**, and Brian turns those equations into simulation code for you. You never write the integration loop yourself: you say what the neurons are, how they are connected and what you want to record, and then call `run()`.

## 0. Setup

To run this notebook on Colab, [click here](https://colab.research.google.com/github/ryanrahy/fcnc_workshop/blob/master/day2/modeling_part1_brian.ipynb).

If you are running this notebook on Google Colab (or any fresh Python environment), run the next cell to install Brian 2. Colab already comes with NumPy and Matplotlib, so `brian2` is the only package we need. If Brian 2 is already installed on your machine, you can skip this cell.

In [ ]:
# get code + data
!git clone https://github.com/ryanrahy/fcnc_workshop.git
import os
os.chdir('./fcnc_workshop/day2/')

# install brian
!pip install brian2

In [ ]:
from brian2 import *
import matplotlib.pyplot as plt
%matplotlib inline

# By default Brian generates and compiles C++ code to make simulations fast.
# Compiling takes a few seconds every time we create a new model, which is longer
# than the small simulations in this notebook take to run, so we tell Brian to
# use plain NumPy instead. Remove this line for large models.
prefs.codegen.target = 'numpy'

## Brian 2 units

Brian uses physical units. Every number in a model carries its physical dimension, and Brian checks that every equation is dimensionally consistent. This catches a whole class of bugs that are otherwise very hard to spot, such as a factor of 1000 lost between milliseconds and seconds, or a current accidentally added to a voltage.

All SI base units are available (`metre`, `kilogram`, `second`, `amp`, `kelvin`, `mole`, `candle`), together with the derived units we need in neuroscience (`volt`, `ohm`, `siemens`, `farad`, `hertz`, ...). The usual prefixes can be attached to any of them: `p` (pico), `n` (nano), `u` (micro), `m` (milli), `k` (kilo), `M` (mega), ... So `mV`, `ms`, `nA`, `pF`, `uS` and `Mohm` all exist.

To give a number a unit, you multiply it by the unit:

In [ ]:
[100*amp, 1e6*volt, 10*nA*5*Mohm]

Brian displays every quantity with a readable prefix: `1e6*volt` is shown as `1. * Mvolt`, and it worked out Ohm's law for us in the last entry ($10\,\mathrm{nA} \times 5\,\mathrm{M\Omega} = 50\,\mathrm{mV}$).

Combining quantities whose dimensions don't match raises a `DimensionMismatchError`:

In [ ]:
try:
    5*mV + 3*nA   # a voltage plus a current makes no sense
except DimensionMismatchError as error:
    print(error)

To get a plain number back out of a quantity, divide it by the unit you want it in. We will use this all the time when plotting, since Matplotlib only understands plain numbers: `monitor.t/ms` gives the recorded times in milliseconds.

In [ ]:
duration = 20*ms
print(duration/ms)       # 20.0
print(duration/second)   # 0.02

____

## 1. Creating neurons

The `NeuronGroup()` function from Brian 2 creates groups of neurons. We will need to define the differential equation for our neuron membrane potential - in this tutorial, we will use the standard equation for a leaky integrate and fire neuron:
$$\tau_m\frac{dV}{dt}=E_L-V+R_mI$$
where
* $V$ is the membrane potential,
* $\tau_m = R_mC_m$ is the membrane time constant, which sets how quickly the membrane responds to its inputs,
* $E_L$ is the resting (leak reversal) potential, the value $V$ settles at when there is no input,
* $R_m$ is the membrane resistance and $I$ the input current. A current $I$ pushes the membrane potential away from rest by $R_mI$.

Without any spiking mechanism, this equation simply says that $V$ relaxes exponentially, with time constant $\tau_m$, towards the value $E_L + R_mI$.

In Brian, the equations are written as a string, with one line per variable. Each line ends with a colon followed by the **unit of the variable** (the unit of $V$, not of $dV/dt$). Lines without a derivative, such as `tau_m : second`, declare **parameters**: every neuron in the group gets its own value, which we set after creating the group. We write $V$ as `v` to follow the Brian convention, and we divide both sides by $\tau_m$.

We will also need to specify a method of integration for our differential equation. Brian can integrate with numerical methods such as `'euler'` or `'rk4'`, but our equation is linear, so it can also be solved analytically, which is what `method='exact'` does. (If you leave out `method`, Brian picks a suitable one and prints which one it chose.) So to create a group of 100 neurons:

In [ ]:
start_scope()   # start a fresh simulation (explained in more detail below)

eq = """
dv/dt = (E_L - v + R_m*I) / tau_m : volt
tau_m : second   # membrane time constant
E_L   : volt     # resting potential
R_m   : ohm      # membrane resistance
I     : amp      # input current
"""

neuron_group = NeuronGroup(100, eq, method='exact')

What are the default parameters of our neurons?

We can read (and set) any variable of a group as an attribute, e.g. `neuron_group.v`. This gives an array with one value per neuron, so we print only the first five here.

In [ ]:
print('v     =', neuron_group.v[:5])
print('tau_m =', neuron_group.tau_m[:5])
print('E_L   =', neuron_group.E_L[:5])
print('R_m   =', neuron_group.R_m[:5])
print('I     =', neuron_group.I[:5])

Every variable starts at **zero**. For `v` this only means a strange starting point, but a membrane time constant of 0 s is meaningless (we would be dividing by zero), so we always have to set the parameters ourselves. We can assign a single value, which every neuron in the group gets, an array with one value per neuron, or a string expression (we'll use one below).

We use $\tau_m$ = 10 ms, $E_L$ = −70 mV, $R_m$ = 100 MΩ and a constant input current of 0.25 nA, and start every neuron at its resting potential:

In [ ]:
neuron_group.tau_m = 10*ms
neuron_group.E_L = -70*mV
neuron_group.R_m = 100*Mohm
neuron_group.I = 0.25*nA
neuron_group.v = -70*mV

print('tau_m =', neuron_group.tau_m[:5])
print('v     =', neuron_group.v[:5])

Brian also has some global defaults. The most important one is the simulation time step `dt` (`defaultclock.dt`), the interval at which the equations are updated, which is 0.1 ms unless you change it.

We want to run a small simulation with this group of neurons and plot our neuron activity. To do so, we need to create objects that monitor the state of our neurons. We can do that with the `StateMonitor` object to monitor our neuron voltages.

`StateMonitor(group, variables, record=...)` records the given variables at every time step. `record` selects which neurons to record from: `True` records all of them, but that quickly uses a lot of memory in bigger simulations, so usually we only pick a few. Here all neurons are identical, so recording neuron 0 is enough.

`run(duration)` then simulates the model. Afterwards, the monitor stores the recording times in `state_mon.t` and the voltages in `state_mon.v`, an array of shape *(number of recorded neurons, number of time steps)*.

In [ ]:
# create a StateMonitor object that records v of neuron 0 at every time step
state_mon = StateMonitor(neuron_group, 'v', record=0)

# run the simulation
run(100*ms)

# plot the outputs
plt.figure(figsize=(8, 3))
plt.plot(state_mon.t/ms, state_mon.v[0]/mV, label='v')
plt.axhline(-45, ls='--', c='gray', label=r'$E_L + R_mI$')
plt.axvline(10, ls=':', c='gray', label=r'$t = \tau_m$')
plt.xlabel('Time (ms)')
plt.ylabel('v (mV)')
plt.legend();

There are no spiking dynamics here; we are simply integrating our differential equation and watching the voltage tend towards $E_L + R_mI$ = −70 mV + 100 MΩ × 0.25 nA = **−45 mV** by the end of the simulation. The approach is exponential: after one time constant (the dotted line at 10 ms) the voltage has covered about 63% of the distance.

To add spiking activity, we set a threshold for our neuron's membrane potential and a refractory period. We also need to create a `SpikeMonitor` to record our neuron spikes.

An integrate-and-fire neuron does not model the shape of the action potential. Instead, when `v` crosses a threshold we simply declare that the neuron has spiked and reset `v`. `NeuronGroup` takes three extra arguments for this:
* `threshold='v > V_th'`: a condition, checked at every time step. When it is true, the neuron emits a spike.
* `reset='v = E_L'`: one or more statements executed right after a spike. Here we put `v` back to the resting potential.
* `refractory=5*ms`: after a spike the neuron cannot spike again for this long.

By default, a refractory neuron only ignores the threshold, while `v` keeps integrating its input. Adding the flag `(unless refractory)` to the end of the equation clamps `v` at its reset value for the whole refractory period, which is the behaviour we usually want.

`SpikeMonitor(group)` records the index `i` of the neuron and the time `t` of every spike.

**`start_scope()`:** when you call `run()`, Brian simulates *every* Brian object you have created (groups, monitors, ...) that is still around. If we now build a new group without clearing the old one, both would be simulated together. `start_scope()` makes Brian forget all objects created before it and resets the simulation time to 0, so we call it at the start of every new simulation.

To make things more interesting, we also give each neuron a different input current, increasing from 0 nA for neuron 0 to 0.3 nA for neuron 99. We do this with a string expression, in which `i` is the index of each neuron and `N` the number of neurons in the group. Similarly, `v = 'E_L'` starts every neuron at its own resting potential.

In [ ]:
start_scope()

# we need to factor refractoriness into our equation: (unless refractory) holds v
# at its reset value while the neuron is refractory
refractory_eq = """
dv/dt = (E_L - v + R_m*I) / tau_m : volt (unless refractory)
tau_m : second   # membrane time constant
E_L   : volt     # resting potential
R_m   : ohm      # membrane resistance
I     : amp      # input current
"""

V_th = -50*mV   # spike threshold, the same for all neurons

neuron_group = NeuronGroup(100, refractory_eq,
                           threshold='v > V_th',  # when does a neuron spike?
                           reset='v = E_L',       # what happens right after a spike?
                           refractory=5*ms,       # how long is it silent afterwards?
                           method='exact')
neuron_group.tau_m = 10*ms
neuron_group.E_L = -70*mV
neuron_group.R_m = 100*Mohm
neuron_group.I = '0.3*nA * i / (N - 1)'   # 0 nA (neuron 0) to 0.3 nA (neuron 99)
neuron_group.v = 'E_L'

# create statemonitor and spikemonitor objects
state_mon = StateMonitor(neuron_group, 'v', record=[25, 75, 99])
spike_mon = SpikeMonitor(neuron_group)

# run
run(100*ms)

# plot the v output w spike times overlayed as lines
spike_trains = spike_mon.spike_trains()   # {neuron index: array of its spike times}
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True, sharey=True)
for ax, v_trace, idx in zip(axes, state_mon.v, state_mon.record):
    ax.plot(state_mon.t/ms, v_trace/mV)
    for t_spike in spike_trains[idx]:
        ax.axvline(t_spike/ms, ls='--', c='r', lw=1)
    ax.axhline(V_th/mV, ls=':', c='gray')
    ax.set_title(f'Neuron {idx} (I = {neuron_group.I[idx]/nA:.2f} nA)')
    ax.set_ylabel('v (mV)')
axes[-1].set_xlabel('Time (ms)')
fig.tight_layout()

The red dashed lines mark the spike times, and the dotted gray line is the threshold. Neuron 25 gets too little current to ever reach threshold: it settles at $E_L + R_mI$ ≈ −62 mV, below $V_{th}$ = −50 mV. The other two neurons charge up to threshold, spike, are reset and held at −70 mV for 5 ms, and charge up again. Neuron 99 gets more current, so it reaches threshold faster and fires more often.

The `SpikeMonitor` recorded the spikes of all 100 neurons, so we can look at the whole group at once. A **raster plot** shows each spike as a dot (time on the x-axis, neuron index on the y-axis). `spike_mon.count` gives the number of spikes of each neuron, from which we get its firing rate. Plotting the firing rate against the input current gives the neuron's **f–I curve**:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(spike_mon.t/ms, spike_mon.i, '.k', ms=2)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Neuron index')
ax1.set_title('Raster plot')

rate = spike_mon.count / (100*ms)   # number of spikes / duration of the simulation
ax2.plot(neuron_group.I/nA, rate/Hz)
ax2.set_xlabel('Input current I (nA)')
ax2.set_ylabel('Firing rate (Hz)')
ax2.set_title('f-I curve')
fig.tight_layout()

Below a certain current the neurons never fire. This threshold current (the *rheobase*) is the current that brings the steady state $E_L + R_mI$ exactly to $V_{th}$, i.e. $I = (V_{th} - E_L)/R_m$ = 20 mV / 100 MΩ = 0.2 nA. Above it, the firing rate increases with the input current. (The curve looks like a staircase because in 100 ms a neuron can only fire a whole number of spikes, so the measured rates come in steps of 10 Hz. A longer simulation would make it smoother.)

Since Brian stores one value per neuron for each parameter, the parameters within a `NeuronGroup` do not need to be identical. We can set different values in different ways:

```python
# one value for every neuron
neuron_group.tau_m = 10*ms

# an array with one value per neuron
neuron_group.tau_m = np.linspace(5, 20, 100)*ms

# a string expression: i is the neuron's index, N the group size, rand() a uniform random number
neuron_group.tau_m = '10*ms + 5*ms*i/(N-1)'
neuron_group.tau_m = '10*ms + 2*ms*randn()'   # normally distributed, different for each neuron

# only some neurons, chosen by index slice or by a condition
neuron_group.tau_m[:50] = 10*ms
neuron_group.tau_m[50:] = 20*ms
neuron_group.tau_m['i % 2 == 0'] = 15*ms      # even-numbered neurons only
```

___

## 2. Connecting neurons

To connect neurons within the same group or between different groups together, we use a `Synapses` object.

First, create two groups of neurons with the same differential equation `refractory_eq` such that:
1. the first group `neuron_group1` has 50 neurons and $\tau_m$ = 50 ms
2. the second group `neuron_group2` has 50 neurons and $\tau_m$ ranging from 20 to 70 ms.

Both groups use the same threshold, reset and refractory period as before (−50 mV, $E_L$ and 5 ms); some of the parameters are set for you. Note that only group 1 receives an input current (0.5 nA). Group 2 gets no current, so any activity in group 2 has to come through its synapses.

<details><summary>Show solution</summary>

```python
neuron_group1 = NeuronGroup(50, refractory_eq, threshold='v > V_th', reset='v = E_L',
                            refractory=5*ms, method='exact')
neuron_group1.tau_m = 50*ms

neuron_group2 = NeuronGroup(50, refractory_eq, threshold='v > V_th', reset='v = E_L',
                            refractory=5*ms, method='exact')
neuron_group2.tau_m = '20*ms + 50*ms*i/(N-1)'   # or: np.linspace(20, 70, 50)*ms
```
</details>

In [ ]:
start_scope()

refractory_eq = """
dv/dt = (E_L - v + R_m*I) / tau_m : volt (unless refractory)
tau_m : second   # membrane time constant
E_L   : volt     # resting potential
R_m   : ohm      # membrane resistance
I     : amp      # input current
"""

V_th = -50*mV   # spike threshold, the same for all neurons

# create the first group
neuron_group1 = ___
# set group parameters
neuron_group1.tau_m = ___
neuron_group1.E_L = -70*mV
neuron_group1.R_m = 100*Mohm
neuron_group1.I = 0.5*nA
neuron_group1.v = 'E_L + rand()*(V_th - E_L)'   # random start, so the neurons don't all fire in sync

# create the second group
neuron_group2 = ___
# set group parameters
neuron_group2.tau_m = ___
neuron_group2.E_L = -70*mV
neuron_group2.R_m = 100*Mohm
neuron_group2.I = 0*nA                           # no input current
neuron_group2.v = 'E_L'

Next, connecting neurons in Brian takes two steps.

**1. Create a `Synapses` object** from a source group to a target group, and say what happens when a presynaptic neuron spikes:
```python
syn = Synapses(source_group, target_group, on_pre='v_post += 1*mV')
```
The `on_pre` statement is executed every time a spike arrives from a presynaptic neuron. In it, the suffix `_post` refers to a variable of the postsynaptic (target) neuron, and `_pre` to the presynaptic (source) neuron. Here, every spike makes the membrane potential of the target neuron jump up by 1 mV. Source and target can be the same group, for connections within a group. By default a spike arrives immediately; adding e.g. `delay=2*ms` makes it arrive later, as it would after travelling along an axon.

**2. Make the connections** with `connect()`. Creating the `Synapses` object alone does not connect any neurons yet:

| connectivity | code |
|---|---|
| all-to-all | `syn.connect()` |
| all-to-all, without self-connections | `syn.connect(condition='i != j')` |
| random, with probability p | `syn.connect(p=0.2)` |
| one-to-one | `syn.connect(j='i')` |
| by index distance, without self-connections | `syn.connect(condition = 'abs(i-j)<4 and i!=j')` |
| a list of pairs | `syn.connect(i=[0, 0, 3], j=[1, 2, 0])` |

In these expressions, `i` is the index of the presynaptic neuron and `j` the index of the postsynaptic neuron, and arguments can be combined (e.g. `condition` and `p`). The last option connects neuron `i[k]` to neuron `j[k]` for every `k`. That is how we will build a network from a connectome in Part 2.

In the next section, you will need to:
1. create the two `Synapses` objects and connect the neurons according to the following:
   1. `syn_11` for connections within group 1: all-to-all minus self-connections with a 20% connection probability, such that every spike raises the target's `v` by 1 mV,
   2. `syn_12` for connections from group 1 to group 2: one-to-one, such that every spike raises the target's `v` by 10 mV.
2. create `SpikeMonitor` objects to record the spikes of both groups.
3. create a `StateMonitor` object to record the voltage of neurons 5 and 45 in group 2.

The rest of the cell runs the network for 1 s and plots the result. Neuron 5 of group 2 has a fast membrane ($\tau_m$ ≈ 25 ms), neuron 45 a slow one ($\tau_m$ ≈ 66 ms).

<details><summary>Show solution</summary>

```python
syn_11 = Synapses(neuron_group1, neuron_group1, on_pre='v_post += 1*mV')
syn_11.connect(condition='i != j', p=0.2)

syn_12 = Synapses(neuron_group1, neuron_group2, on_pre='v_post += 10*mV')
syn_12.connect(j='i')

spike_mon1 = SpikeMonitor(neuron_group1)
spike_mon2 = SpikeMonitor(neuron_group2)
state_mon2 = StateMonitor(neuron_group2, 'v', record=[5, 45])
```
</details>

In [ ]:
# create the synapses within group 1
syn_11 = ___
# connect them
syn_11.connect(___)

# create the synapses from group 1 to 2
syn_12 = ___
# connect them
syn_12.connect(___)
print(f'{len(syn_11)} synapses within group 1, {len(syn_12)} from group 1 to group 2')

# record spikes from both groups, and v of two neurons in group 2
spike_mon1 = ___
spike_mon2 = ___
state_mon2 = ___

run(1*second)


def plot_network(spike_mon1, spike_mon2, state_mon2):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    t_start = state_mon2.t[0]   # start of this run
    duration = state_mon2.t[-1] - t_start + defaultclock.dt

    # raster plot: group 1 at the bottom (0-49), group 2 on top (50-99)
    axes[0].plot((spike_mon1.t - t_start)/ms, spike_mon1.i, '.k', ms=2)
    axes[0].plot((spike_mon2.t - t_start)/ms, spike_mon2.i + 50, '.', c='tab:orange', ms=2)
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Neuron index')
    axes[0].set_title('Raster plot (black: group 1, orange: group 2)')

    # voltage of the recorded group 2 neurons during the first 300 ms
    keep = state_mon2.t < t_start + 300*ms
    for v_trace, idx in zip(state_mon2.v, state_mon2.record):
        axes[1].plot((state_mon2.t[keep] - t_start)/ms, v_trace[keep]/mV,
                     label=f'neuron {idx} ($\\tau_m$ = {neuron_group2.tau_m[idx]/ms:.0f} ms)')
    axes[1].axhline(V_th/mV, ls=':', c='gray')
    axes[1].set_xlabel('Time (ms)')
    axes[1].set_ylabel('v (mV)')
    axes[1].set_title('Group 2 voltage')
    axes[1].legend(loc='lower right')

    # firing rate of every group 2 neuron against its membrane time constant
    axes[2].plot(neuron_group2.tau_m/ms, spike_mon2.count/duration/Hz, 'o-', c='tab:orange', ms=3)
    axes[2].set_xlabel(r'$\tau_m$ of the group 2 neuron (ms)')
    axes[2].set_ylabel('Firing rate (Hz)')
    axes[2].set_title('Group 2 firing rates')
    fig.tight_layout()
    print(f'mean firing rate: group 1 {spike_mon1.num_spikes/50/duration/Hz:.1f} Hz, '
          f'group 2 {spike_mon2.num_spikes/50/duration/Hz:.1f} Hz')


plot_network(spike_mon1, spike_mon2, state_mon2)

Group 2 has no input current of its own, so every group 2 spike is caused by spikes from group 1. Yet not every group 2 neuron fires, even though they all receive the same kind of input: one partner firing at about 35 Hz, each spike worth 10 mV. Only neurons with a slow membrane ($\tau_m$ above roughly 40 ms) reach threshold. The voltage traces show why. Every input spike makes `v` jump by 10 mV, and the jump then decays with the membrane time constant. In the fast neuron, most of it has decayed before the next spike arrives. In the slow neuron, consecutive jumps add up (**temporal summation**) until `v` crosses threshold. The membrane time constant sets how long a neuron "remembers" its inputs.

You may also notice that group 1 synchronizes within the first ~100 ms: its recurrent excitatory connections pull the neurons into firing together.

We can also set weights to synaptic connections:

So far, all synapses of a `Synapses` object had the same effect. To give each synapse its own strength, we declare a **synaptic variable** `w` in the `model` argument and use it in `on_pre`:
```python
syn = Synapses(source_group, target_group, model='w : volt', on_pre='v_post += w')
```
Once the synapses are connected, `syn.w` holds one value per synapse. We set it just like a neuron parameter: a single value, a string expression (which can use `i`, `j` and `rand()`), or an array with one value per synapse. The arrays `syn.i` and `syn.j` hold the presynaptic and postsynaptic neuron of every synapse, in the same order as `syn.w`. A **negative** weight makes a synapse **inhibitory**: a spike pushes `v` down instead of up.

Let's turn our network into a small "connectome":
* Every connection within group 1 consists of a random number of synapses (1 to 5), each worth 0.5 mV.
* Neurons 40–49 of group 1 are inhibitory, so all their outgoing synapses get a negative sign. Following Dale's law, the sign belongs to the presynaptic neuron, as in the signed adjacency matrix of FlyWire intro Part 3.

This is how we will build the model of the fly brain in Part 2: weight = sign of the presynaptic neuron × number of synapses × weight per synapse.

We start a new simulation with the same two groups as above, and create the synapses with weights.

In [ ]:
start_scope()

# the same two groups as above
neuron_group1 = NeuronGroup(50, refractory_eq, threshold='v > V_th', reset='v = E_L',
                            refractory=5*ms, method='exact')
neuron_group1.tau_m = 50*ms
neuron_group1.E_L = -70*mV
neuron_group1.R_m = 100*Mohm
neuron_group1.I = 0.5*nA
neuron_group1.v = 'E_L + rand()*(V_th - E_L)'

neuron_group2 = NeuronGroup(50, refractory_eq, threshold='v > V_th', reset='v = E_L',
                            refractory=5*ms, method='exact')
neuron_group2.tau_m = '20*ms + 50*ms*i/(N-1)'
neuron_group2.E_L = -70*mV
neuron_group2.R_m = 100*Mohm
neuron_group2.I = 0*nA
neuron_group2.v = 'E_L'

sign = np.ones(50)
sign[40:] = -1   # neurons 40-49 of group 1 are inhibitory

# within group 1: sign of the presynaptic neuron x number of synapses x 0.5 mV
syn_11 = Synapses(neuron_group1, neuron_group1, model='w : volt', on_pre='v_post += w')
syn_11.connect(condition='i != j', p=0.2)
n_synapses = np.random.randint(1, 6, size=len(syn_11))   # 1 to 5 synapses per connection
syn_11.w = sign[syn_11.i[:]] * n_synapses * 0.5*mV

# group 1 -> group 2: sign of the presynaptic neuron x 10 mV
syn_12 = Synapses(neuron_group1, neuron_group2, model='w : volt', on_pre='v_post += w')
syn_12.connect(j='i')
syn_12.w = sign[syn_12.i[:]] * 10*mV

print('first five weights within group 1:', syn_11.w[:5])

spike_mon1 = SpikeMonitor(neuron_group1)
spike_mon2 = SpikeMonitor(neuron_group2)
state_mon2 = StateMonitor(neuron_group2, 'v', record=[5, 45])

run(1*second)

plot_network(spike_mon1, spike_mon2, state_mon2)

Neurons 40–49 of group 2, which fired most before, are now silent, because their only input comes from an inhibitory neuron. In the voltage trace of neuron 45, every input spike now pushes `v` *down*. It even drops to about −100 mV, far below the resting potential, which a real neuron can't do: an inhibitory synapse can only pull `v` towards its reversal potential (around −80 mV). Conductance-based synapses (see below) capture this. The other group 2 neurons still fire. Group 1's overall firing rate barely changes: its excitatory connections are stronger on average (1.5 mV instead of 1 mV per spike), but some of its neurons now also receive inhibition.

There are more complex synapse models but these are beyond the scope of this workshop.

In our synapses, a spike makes `v` jump instantly. Real synaptic input is spread out over a few milliseconds: a spike opens ion channels, and the resulting synaptic current rises and decays again. A common way to model this is with a **synaptic variable** that jumps when a spike arrives and decays exponentially, and that drives the membrane potential:
```python
eq = """
dv/dt = (E_L - v + g) / tau_m : volt
dg/dt = -g / tau_syn          : volt   # synaptic input, decays with tau_syn
"""
syn = Synapses(source_group, target_group, model='w : volt', on_pre='g_post += w')
```
The Shiu et al. model in Part 2 uses exactly this kind of synapse. Other common extensions:
* **Conductance-based synapses:** the effect of a spike depends on how far `v` is from the synapse's reversal potential, as in real channels.
* **Short-term plasticity:** synapses get weaker (depression) or stronger (facilitation) when they are used repeatedly within a short time.
* **Long-term plasticity**, e.g. **spike-timing-dependent plasticity (STDP)**: the weight `w` itself changes depending on the relative timing of pre- and postsynaptic spikes. If the presynaptic neuron fires just before the postsynaptic one, the synapse gets stronger; if it fires just after, it gets weaker. This is a model of learning. In Brian, this is done by adding equations for the plasticity variables to the `Synapses` model and an `on_post` statement for postsynaptic spikes (see the [STDP section of the Brian synapses tutorial](https://brian2.readthedocs.io/en/stable/resources/tutorials/2-intro-to-brian-synapses.html)).

____

## 3. Stimulating neurons

So far, we drove our neurons with a constant current, which makes them fire perfectly regularly. Real neurons receive irregular input from many other neurons, and in experiments we often stimulate a chosen set of neurons, for example with optogenetics. A common way to model both is with **Poisson spike trains**: spikes that occur at random times, with a given average rate. Brian has two tools for this:

* `PoissonGroup(N, rates=...)` creates `N` "neurons" that spike randomly at the given rate. We connect them to other neurons with `Synapses`, just like a `NeuronGroup`.
* `PoissonInput(target, target_var, N, rate, weight)` is a shortcut for when we don't need the individual input neurons. Every neuron in `target` gets `N` independent Poisson inputs at frequency `rate`, and every input spike adds `weight` to the variable `target_var`. This is much faster than simulating many input neurons and synapses.

Below, we use both on a group of 50 neurons without an input current:
1. **Background input:** every neuron receives `PoissonInput` from 200 inputs at 10 Hz, each spike worth 1 mV. This keeps `v` fluctuating around the threshold, so the neurons fire irregularly.
2. **"Optogenetic" stimulation:** neurons 0–9 also receive spikes from a `PoissonGroup` firing at 50 Hz, connected one-to-one. Each of these spikes is worth 30 mV, enough to make the neuron fire right away. This is how Shiu et al. activate neurons in their model of the fly brain (Part 2).

In [ ]:
start_scope()

neuron_group = NeuronGroup(50, refractory_eq, threshold='v > V_th', reset='v = E_L',
                           refractory=5*ms, method='exact')
neuron_group.tau_m = 10*ms
neuron_group.E_L = -70*mV
neuron_group.R_m = 100*Mohm
neuron_group.I = 0*nA
neuron_group.v = 'E_L'

# 1. background: 200 Poisson inputs at 10 Hz per neuron, each adding 1 mV to v
background = PoissonInput(neuron_group, 'v', N=200, rate=10*Hz, weight=1*mV)

# 2. stimulation: 10 Poisson neurons at 50 Hz, connected one-to-one to neurons 0-9
stimulus = PoissonGroup(10, rates=50*Hz)
stim_syn = Synapses(stimulus, neuron_group, on_pre='v_post += 30*mV')
stim_syn.connect(j='i')

spike_mon = SpikeMonitor(neuron_group)
state_mon = StateMonitor(neuron_group, 'v', record=[5, 30])
run(500*ms)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
ax1.plot(spike_mon.t/ms, spike_mon.i, '.k', ms=3)
ax1.axhspan(-0.5, 9.5, color='tab:blue', alpha=0.15, label='stimulated')
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Neuron index')
ax1.set_title('Raster plot')
ax1.legend(loc='upper right')

keep = state_mon.t < 200*ms
for v_trace, idx in zip(state_mon.v, state_mon.record):
    ax2.plot(state_mon.t[keep]/ms, v_trace[keep]/mV, label=f'neuron {idx}')
ax2.axhline(V_th/mV, ls=':', c='gray')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('v (mV)')
ax2.set_title('Voltage')
ax2.legend(loc='lower right')
fig.tight_layout()

rates = spike_mon.count / (500*ms)
print(f'mean rate: stimulated {rates[:10].mean()/Hz:.0f} Hz, others {rates[10:].mean()/Hz:.0f} Hz')

With only the background input, neuron 30 fluctuates just below threshold and fires whenever a fluctuation crosses it, at irregular times. Neuron 5 receives the same background, plus a strong spike every 20 ms on average, so it fires more often. In the raster, the stimulated neurons stand out at the bottom.